In [ ]:
!pip install transformers datasets accelerate scikit-learn pandas -q

In [ ]:
import torch
print(torch.__version__)  # just to confirm torch is working

In [ ]:
!pip uninstall transformers -y
!pip install transformers==4.44.0 -q

In [ ]:
import torch
from transformers import BertForSequenceClassification
import transformers

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)
print("Model loaded successfully!")

In [ ]:
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score

from google.colab import files
uploaded = files.upload()

df = pd.read_csv("sentiment_data.csv")


if "sentiment" in df.columns:
   df = df.rename(columns={"sentiment": "labels"})

df["labels"] = df["labels"].map({"positive": 1, "negative": 0})

df = df.dropna(subset=["labels"])
df["labels"] = df["labels"].astype(int)

dataset = Dataset.from_pandas(df)
dataset = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset["train"]
test_dataset = dataset["test"]

tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize_function(example):
    return tokenizer(
        example["review"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
test_dataset = test_dataset.map(tokenize_function, batched=True)

train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

model = BertForSequenceClassification.from_pretrained("bert-base-uncased", num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    accuracy = accuracy_score(labels, predictions)
    return {"accuracy": accuracy}

training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    learning_rate=2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

results = trainer.evaluate()
print(results)

model.save_pretrained("bert_sentiment_model")
tokenizer.save_pretrained("bert_sentiment_model")



In [ ]:
review = "Wow, just wow! If your idea of a cinematic masterpiece is watching paint dry in slow motion while listening to nails scratching on a chalkboard, then this movie is an absolute triumph. The director deserves an award for managing to make a 90-minute film feel like an eternity. Truly a breathtaking achievement in testing human endurance"
review = review.encode("ascii", "ignore").decode()
inputs = tokenizer(review, return_tensors="pt", truncation=True, padding=True)
inputs = {k: v.to(model.device) for k, v in inputs.items()}  #moves all input tensors to the same device as the model (CPU or GPU)

model.eval()
with torch.no_grad():
    outputs = model(**inputs)

prediction = torch.argmax(outputs.logits, dim=-1).item()

if prediction == 1:
    print("\nSentiment: Positive Review")
else:
    print("\nSentiment: Negative Review")